# 方法汇总、新数据对比与数据量建议

## 说明

本 Notebook 汇总所有方法的真实结果，包含两轮数据集：
- **V1（原始数据）**：481 个片段，每类约 48 个
- **V2（补充数据后）**：1175 个片段，每类约 118 个（新增约 4.9 倍 base 数据）

**所有准确率均为 5 个随机种子的平均值 ± 标准差**，避免单次划分的运气成分。

## 核心结论

| 方法 | V1 (481样本) | V2 (1175样本) |
|------|------------|______________|
| 基线 9D+RF | 0.667 | — |
| 路线A VMD+ET | 0.768 | — |
| 路线B CWT+CNN | 0.747 | — |
| 增强特征 232D+ET | 0.796±0.054 | **0.816±0.018** |
| Stacking(ET+SVM+RF) | — | **0.842±0.021** |
| Stacking+文件投票 | — | **0.853±0.020** |

增加数据后准确率从 0.796 提升到 0.842，方差从 ±0.054 大幅收窄到 ±0.021。
**剩余瓶颈**：手势 4 和 sc 的准确率仍低（约 0.5），是整体达不到 90% 的唯一原因。

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

r1 = json.load(open(PROJECT_ROOT / 'data' / 'processed' / 'method_results.json'))
r2 = json.load(open(PROJECT_ROOT / 'data' / 'processed' / 'method_results_v2.json'))

print(f"V1 数据集: {r1['n_samples']} 个片段")
print(f"V2 数据集: {r2['n_samples']} 个片段（补充数据后）")

---
## 1. 所有方法准确率对比

以下汇总了从基线到最终 Stacking 集成的所有方法。

In [ ]:
from src.preprocess.io import GESTURE_NAMES

methods = [
    ('Baseline 9D+RF',         0.667,                            0,                            'V1', '#9E9E9E'),
    ('Route A VMD+ET',         0.768,                            0,                            'V1', '#90CAF9'),
    ('Route B CWT+CNN',        0.747,                            0,                            'V1', '#FFCC80'),
    ('Enhanced 232D+ET (V1)',  r1['enhanced_seg']['mean'],       r1['enhanced_seg']['std'],    'V1', '#66BB6A'),
    ('Enhanced 232D+ET (V2)',  r2['enhanced_seg']['mean'],       r2['enhanced_seg']['std'],    'V2', '#26A69A'),
    ('Stacking ET+SVM+RF',     r2['stacking_seg']['mean'],       r2['stacking_seg']['std'],    'V2', '#1565C0'),
    ('Stacking+File Vote',     r2['stacking_vote']['mean'],      r2['stacking_vote']['std'],   'V2', '#AD1457'),
]

fig, ax = plt.subplots(figsize=(13, 6))
for i, (name, acc, std, ver, color) in enumerate(methods):
    ax.bar(i, acc, yerr=std if std > 0 else None, capsize=4,
           color=color, edgecolor='white', width=0.7)
    label_text = f'{acc:.3f}' + (f'  +-{std:.3f}' if std > 0 else '')
    ax.text(i, acc + (std if std > 0 else 0) + 0.015, label_text,
            ha='center', fontsize=8, fontweight='bold')

ax.axhline(0.90, color='red', linestyle='--', linewidth=1.5, label='Target 90%')
ax.axhline(0.10, color='gray', linestyle=':', linewidth=0.8, label='Random 10%')
ax.set_xticks(range(len(methods)))
ax.set_xticklabels([m[0] for m in methods], fontsize=8, rotation=20, ha='right')
ax.set_ylabel('Test Accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('All Methods: Test Accuracy Comparison (error bars = 5-seed std)', fontsize=12)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')
ax.axvline(3.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax.text(1.5, 0.98, 'Original data (V1)', ha='center', fontsize=9, color='gray')
ax.text(5.0, 0.98, 'With extra data (V2)', ha='center', fontsize=9, color='#1565C0')
plt.tight_layout()
plt.show()

### 如何读这张图

- 左侧 4 根柱子是使用原始数据（V1，481样本）的结果
- 右侧 3 根柱子是补充数据后（V2，1175样本）的结果
- **误差线**是 5 种子标准差——V2 的误差线更短，说明结果更稳定
- **红色虚线**是 90% 目标，目前最好方法（Stacking+投票）达到 0.853

---
## 2. 数据增加的效果：V1 vs V2 直接对比

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

vals = [r1['enhanced_seg']['mean'], r2['enhanced_seg']['mean']]
stds = [r1['enhanced_seg']['std'],  r2['enhanced_seg']['std']]
labels = ['V1 (481)', 'V2 (1175)']
colors = ['#66BB6A', '#26A69A']
bars = axes[0].bar(labels, vals, yerr=stds, capsize=6, color=colors, edgecolor='white', width=0.5)
for bar, v, s in zip(bars, vals, stds):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+s+0.015,
                 f'{v:.3f}  +-{s:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[0].axhline(0.90, color='red', linestyle='--', alpha=0.6, label='90%')
axes[0].set_ylabel('Test Accuracy (Enhanced+ET)')
axes[0].set_title('Accuracy: V1 vs V2')
axes[0].set_ylim(0, 1.0); axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

axes[1].barh(['V2 (new data)', 'V1 (original)'], [1175, 481],
             color=['#26A69A','#66BB6A'], edgecolor='white')
axes[1].set_xlabel('Number of Segments')
axes[1].set_title('Dataset Size Comparison')
for i, n in enumerate([1175, 481]):
    axes[1].text(n+20, i, str(n), va='center', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(f'V1->V2 准确率提升: +{(vals[1]-vals[0])*100:.1f}%')
print(f'V1->V2 标准差收窄: {stds[0]:.3f} -> {stds[1]:.3f}')
print(f'V1->V2 数据量: 481 -> 1175 ({1175/481:.1f}x)')

---
## 3. 各手势准确率：剩余瓶颈

In [ ]:
per_g_v2 = r2['per_gesture']
per_g_v1 = r1['per_gesture']
gestures = [GESTURE_NAMES[int(k)] for k in sorted(per_g_v2.keys(), key=int)]
v2_accs = [per_g_v2[k] for k in sorted(per_g_v2.keys(), key=int)]
v1_accs = [per_g_v1.get(k, 0) for k in sorted(per_g_v2.keys(), key=int)]

x = np.arange(len(gestures))
w = 0.38
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x-w/2, v1_accs, w, label='V1 (481)', color='#90CAF9', edgecolor='white')
b2 = ax.bar(x+w/2, v2_accs, w, label='V2 (1175)', color='#1565C0', edgecolor='white')
ax.axhline(0.90, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(gestures, rotation=45)
ax.set_ylabel('Accuracy (5-seed avg)'); ax.set_ylim(0, 1.1)
ax.set_title('Per-gesture Accuracy: V1 vs V2')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
for b, v in zip(b2, v2_accs):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.2f}',
            ha='center', fontsize=8, fontweight='bold')
plt.tight_layout(); plt.show()

print('V2 各手势准确率:')
for g, a in zip(gestures, v2_accs):
    status = 'OK' if a >= 0.85 else 'WARN' if a >= 0.70 else 'LOW'
    print(f'  [{status}] {g:12s}: {a:.2f}')

### 手势 4 和 sc 是唯一瓶颈

其他 8 个手势在 V2 中已达 0.76-1.00。手势 4（~0.55）和 sc（~0.50）
的 TENG 三通道信号与其他手势高度重叠，是整体准确率无法到 90% 的根本原因。
增加这两个手势的多样性数据（不同人、不同姿势）是唯一出路。

---
## 4. 学习曲线：数据量与准确率的关系

In [ ]:
lc1 = r1['learning_curve']
lc2 = r2['learning_curve']

fig, ax = plt.subplots(figsize=(11, 6))

n1 = [lc1[k]['n_train'] for k in sorted(lc1, key=float)]
a1 = [lc1[k]['mean']    for k in sorted(lc1, key=float)]
s1 = [lc1[k]['std']     for k in sorted(lc1, key=float)]
ax.errorbar(n1, a1, yerr=s1, marker='o', linewidth=2, capsize=4,
            color='#66BB6A', label='V1 measured')

n2 = [lc2[k]['n_train'] for k in sorted(lc2, key=float)]
a2 = [lc2[k]['mean']    for k in sorted(lc2, key=float)]
s2 = [lc2[k]['std']     for k in sorted(lc2, key=float)]
ax.errorbar(n2, a2, yerr=s2, marker='s', linewidth=2, capsize=4,
            color='#1565C0', label='V2 measured')

ex2 = r2['extrapolation']
a_fit, b_fit = ex2['fit_a'], ex2['fit_b']
N_ext = np.linspace(200, 4000, 300)
ax.plot(N_ext, a_fit*np.log(N_ext)+b_fit, '--', color='#FF7043',
        alpha=0.8, linewidth=1.5, label='V2 extrapolation')

N_90 = ex2['targets']['0.90']['n_train']
ax.axhline(0.90, color='red', linestyle=':', alpha=0.5)
ax.axvline(N_90, color='red', linestyle=':', alpha=0.3)
ax.annotate(f'90%: ~{N_90} samples',
            xy=(N_90, 0.90), xytext=(N_90+300, 0.84),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red', alpha=0.5))
ax.set_xlabel('Number of Training Segments')
ax.set_ylabel('Test Accuracy (5-seed avg)')
ax.set_title('Learning Curve: V1 vs V2')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0.55, 1.0)
plt.tight_layout(); plt.show()

print('V2 学习曲线 (학습곡선):')
for k in sorted(lc2, key=float):
    print(f"  {lc2[k]['n_train']:5d} 片段: {lc2[k]['mean']:.3f} +/- {lc2[k]['std']:.3f}")

### 如何读这张图

V2 学习曲线在 700-940 片段之间几乎持平（0.817→0.816），
说明**当前数据量已接近手势 4/sc 的可区分性上限**。
继续加同类数据收益有限，需要提高数据多样性（不同采集者、不同姿势）。

---
## 5. 数据量建议

In [ ]:
ex2 = r2['extrapolation']
a_fit, b_fit = ex2['fit_a'], ex2['fit_b']

print('基于V2学习曲线外推:')
print(f"当前训练片段: {ex2['current_train']} (每类约 {ex2['current_per_class']})")
print()
for t_str, label in [('0.85', '85% (ET单片段)'), ('0.87', '87% (ET+Stacking~90%)'), ('0.90', '90% (ET单片段)')]:
    N = int(round( (float(t_str) - b_fit) / a_fit ))
    N = max(N, 1)
    import math
    N = int(math.exp( (float(t_str) - b_fit) / a_fit ))
    mult = round(N / ex2['current_train'], 1)
    print(f'  达到 {label}: ~{N} 训练片段 (每类~{N//10}, 现在的{mult}倍)')

print()
print('关键建议:')
print('  1. 手势4和sc是唯一瓶颈，重点补采这两个手势')
print('  2. 补采时要多样性：不同人、不同姿势、不同时间')
print('  3. 三种环境(base/wind_noise/uv_radiation)都要补')
print('  4. 补采后直接运行 scripts/01_run_pipeline.py 重跑即可')

### 数据量建议总结

| 目标 | 需训练片段 | 每类 | 相比现在 | 预期整体准确率 |
|------|----------|------|---------|---------------|
| ET 单片段 85% | ~1100 | ~110 | 1.2x | — |
| **ET+Stacking 约 90%** | **~1400** | **~140** | **1.5x** | **~90%** |
| ET 单片段 90% | ~2200 | ~220 | 2.4x | ~93% |

**最务实方案**：对手势 4 和 sc 各补采 3-4 个文件（每文件 20 次动作），
其他手势不需要追加。补采完运行流水线即可，代码不用修改。

---
## 6. 最终总结

In [ ]:
print('='*60)
print('FINAL SUMMARY')
print('='*60)
print(f"V1: {r1['n_samples']} 片段  V2: {r2['n_samples']} 片段")
print()
rows = [
    ('Baseline 9D+RF',          '0.667', '—'),
    ('Route A VMD+ET',           '0.768', '—'),
    ('Route B CWT+CNN',          '0.747', '—'),
    ('Enhanced 232D+ET (V1)',    f"{r1['enhanced_seg']['mean']:.3f}+/-{r1['enhanced_seg']['std']:.3f}", '—'),
    ('Enhanced 232D+ET (V2)',    '—', f"{r2['enhanced_seg']['mean']:.3f}+/-{r2['enhanced_seg']['std']:.3f}"),
    ('Stacking ET+SVM+RF (V2)', '—', f"{r2['stacking_seg']['mean']:.3f}+/-{r2['stacking_seg']['std']:.3f}"),
    ('Stacking+File Vote (V2)',  '—', f"{r2['stacking_vote']['mean']:.3f}+/-{r2['stacking_vote']['std']:.3f}"),
]
print(f'  {"Method":<28} {"V1":>14} {"V2":>14}')
print('  ' + '-'*58)
for name, v1, v2 in rows:
    print(f'  {name:<28} {v1:>14} {v2:>14}')
print()
print('Bottleneck: gesture 4 (~0.55) and sc (~0.50)')
print('Action: add 3-4 more files for gesture 4 and sc only')